In [12]:
import sys
from pathlib import Path
import re
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy import stats
from ipywidgets import interact, Dropdown, IntSlider, SelectionRangeSlider
sys.path.insert(0, '../')
from ultility.metrics import rolling_std_vol
from ultility.eval_vol import compute_metrics_from_preds

In [13]:
datasets = {}
for csv_name, key in [('VN30_INDEX.csv', 'VN30 Index'), ('VN_INDEX.csv', 'VN Index')]:
    df_temp = pd.read_csv(f'../dataset/{csv_name}')
    df_temp['time'] = pd.to_datetime(df_temp['time'], format='mixed', dayfirst=True, errors='coerce')
    df_temp = df_temp.sort_values('time')
    ret_col = 'return_1_day'
    df_temp_filtered = df_temp[df_temp['time'].dt.year >= 2010]
    null_count = df_temp_filtered[ret_col].isna().sum()
    if null_count > 0:
        for idx, val in df_temp_filtered[df_temp_filtered[ret_col].isna()][ret_col].items():
            print(f"{key}: null at {df_temp.loc[idx, 'time'].strftime('%Y-%m-%d')}")
    datasets[key] = df_temp_filtered.set_index('time')[ret_col] * 100

for file in ['DAX_40.csv', 'EuroNext_100.csv', 'IBEX_35.csv', 'KOSPI_index.csv', 'SMI.csv', 'snp500.csv', 'Nikkei_225.csv']:
    df_temp = pd.read_csv(f'../dataset/{file}')
    if 'Date' in df_temp.columns:
        df_temp.rename(columns={'Date': 'time'}, inplace=True)
    df_temp['time'] = pd.to_datetime(df_temp['time'], format='mixed', dayfirst=False, errors='coerce')
    df_temp = df_temp.dropna(subset=['time']).sort_values('time')
    ret_col = 'return_1_day'
    df_temp_filtered = df_temp[df_temp['time'].dt.year >= 2010]
    null_count = df_temp_filtered[ret_col].isna().sum()
    if null_count > 0:
        for idx, val in df_temp_filtered[df_temp_filtered[ret_col].isna()][ret_col].items():
            print(f"{file}: null at {df_temp.loc[idx, 'time'].strftime('%Y-%m-%d')}")
    datasets[file.replace('.csv', '')] = df_temp_filtered.set_index('time')[ret_col] * 100


In [14]:
summary_split = pd.DataFrame({
    "Dataset": [
        "VN30 Index", "VN Index", "DAX_40", "EuroNext_100", "IBEX_35", "KOSPI_index", "SMI", "snp500", "Nikkei_225"
    ],
    "train_size": [1971, 1964, 1983, 2005, 1815, 1944, 1987, 1988, 1677],
    "val_size":   [1096, 1103, 1104, 1115, 1362, 1109, 1135, 1109, 1174],
    "test_size":  [925, 925, 972, 979, 923, 881, 901, 927, 1062],
    "split_i":    [1971, 1964, 1983, 2005, 1815, 1944, 1987, 1988, 1677],
    "split_j":    [3067, 3067, 3087, 3120, 3177, 3053, 3122, 3097, 2851]
})
split_df = summary_split.set_index("Dataset")[["train_size", "val_size", "test_size"]]
split_df.columns = ["Train", "Val", "Test"]
split_df

,Train,Val,Test
Dataset,,,
VN30 Index,1971,1096,925
VN Index,1964,1103,925
DAX_40,1983,1104,972
EuroNext_100,2005,1115,979
IBEX_35,1815,1362,923
KOSPI_index,1944,1109,881
SMI,1987,1135,901
snp500,1988,1109,927
Nikkei_225,1677,1174,1062


In [15]:
def calc_vol(x, typ, window):
    if typ == "abs(return)":
        return x.abs()
    if typ == "vol(rolling)":
        return x.rolling(window).std()
    return x.rolling(window).apply(lambda y: (y**2).mean()**0.5, raw=True)

def plot_chart(ds, chart_type, typ, period, window):
    try:
        series = datasets[ds]
        tr, va, te = map(int, split_df.loc[ds, ['Train', 'Val', 'Test']])
        fig = go.Figure()
        if chart_type == "Volatility":
            full_v = calc_vol(series, typ, window)
            if period == 'all':
                for s, e, c, n in [(0, tr, '#1f77b4', 'train'), (tr, tr+va, '#ff7f0e', 'val'), (tr+va, tr+va+te, '#2ca02c', 'test')]:
                    fig.add_trace(go.Scatter(x=series.index[s:e], y=full_v.iloc[s:e], mode="lines", name=n, line=dict(color=c), connectgaps=False))
            else:
                s, e = (0, tr) if period == 'train' else (tr, tr+va) if period == 'val' else (tr+va, tr+va+te)
                fig.add_trace(go.Scatter(x=series.index[s:e], y=full_v.iloc[s:e], mode="lines", name=period, line=dict(color="#1f77b4"), connectgaps=False))
            fig.update_layout(title=f"{ds} - {typ} ({period}, window={window})", xaxis_title="Time", yaxis_title="Volatility")
        else:
            if period == 'all':
                for s, e, n in [(0, tr, 'train'), (tr, tr+va, 'val'), (tr+va, tr+va+te, 'test')]:
                    fig.add_trace(go.Histogram(x=series.iloc[s:e], nbinsx=50, name=n))
            else:
                s, e = (0, tr) if period == 'train' else (tr, tr+va) if period == 'val' else (tr+va, tr+va+te)
                fig.add_trace(go.Histogram(x=series.iloc[s:e], nbinsx=50, name=ds))
            fig.update_layout(title=f"{ds} - Return Distribution ({period})", xaxis_title="Return (%)", yaxis_title="Frequency", xaxis=dict(range=[-16, 16]))
        fig.update_layout(height=500)
        fig.show()
    except Exception as e:
        print(f"Error: {e}")

ds_opts = [k for k in split_df.index if k in datasets]
interact(
    plot_chart,
    ds=Dropdown(options=ds_opts),
    chart_type=Dropdown(options=["Volatility", "Histogram"], value="Volatility"),
    typ=Dropdown(options=["abs(return)", "vol(rolling)", "realized_vol"], value="vol(rolling)"),
    period=Dropdown(options=["train", "val", "test", "all"], value="train"),
    window=Dropdown(options=[10, 20, 30, 50, 60, 120, 250], value=60)
)

interactive(children=(Dropdown(description='ds', options=('VN30 Index', 'VN Index', 'DAX_40', 'EuroNext_100', …

<function __main__.plot_chart(ds, chart_type, typ, period, window)>

In [16]:
dist_df = pd.DataFrame([
    {'Dataset': ds, 'nu_train': stats.t.fit(datasets[ds].iloc[:int(split_df.loc[ds, 'Train'])].dropna().values)[0]}
    for ds in split_df.index if ds in datasets
])
dist_df

,Dataset,nu_train
0,VN30 Index,4.696197
1,VN Index,4.573257
2,DAX_40,3.424473
3,EuroNext_100,3.502448
4,IBEX_35,4.400113
5,KOSPI_index,3.426984
6,SMI,3.589687
7,snp500,2.705923
8,Nikkei_225,4.926478


In [ ]:
preds_df = pd.read_csv('../output/predicts.csv')
nu_dict = dict(zip(dist_df['Dataset'], dist_df['nu_train']))
metrics_df = compute_metrics_from_preds(preds_df, datasets, split_df, nu_dict, seq_len=60)
metrics_df.to_csv('../output/metrics.csv', index=False)

In [9]:
preds_raw = pd.read_csv('../output/predicts.csv')
preds_raw = preds_raw[preds_raw['Model'] != 'TransformerGARCH'].copy()
preds_raw['pred_vol'] = pd.to_numeric(preds_raw['predicted_vol'], errors='coerce')
preds_raw['Date_t'] = pd.to_datetime(preds_raw['Date'], errors='coerce', format='mixed', dayfirst=True)
preds_raw['Date_f'] = pd.to_datetime(preds_raw['Date'], errors='coerce', format='mixed', dayfirst=False)

def _best_slice(raw, exp_dates):
    best = None
    for c in ['Date_t', 'Date_f']:
        x = raw[['pred_vol', c]].dropna().sort_values(c).reset_index(drop=True)
        n = min(len(x), len(exp_dates))
        if n == 0:
            continue
        d = pd.to_datetime(x[c]).to_numpy()
        score_best, off_best = -1, 0
        for off in range(len(x) - n + 1):
            score = pd.Index(d[off:off + n]).isin(exp_dates).sum()
            if score > score_best:
                score_best, off_best = score, off
        if best is None or score_best > best[0]:
            best = (score_best, c, off_best, x.iloc[off_best:off_best + n].reset_index(drop=True))
    return best

fixed_rows, fix_sum = [], []
for ds in split_df.index:
    tr, va, te = map(int, split_df.loc[ds, ['Train', 'Val', 'Test']])
    exp_dates = pd.to_datetime(datasets[ds].index[tr + va + 60:tr + va + te])
    test_data = datasets[ds].iloc[tr + va:tr + va + te].values
    rv = rolling_std_vol(test_data, 60)[60:]
    ret = test_data[60:]
    for model in preds_raw.loc[preds_raw['Dataset'] == ds, 'Model'].dropna().unique():
        raw = preds_raw[(preds_raw['Dataset'] == ds) & (preds_raw['Model'] == model)]
        best = _best_slice(raw, exp_dates)
        if best is None:
            continue
        score, parse_used, off, sl = best
        n = min(len(exp_dates), len(sl))
        fixed_rows.append(pd.DataFrame({
            'Dataset': ds,
            'Model': model,
            'Date': exp_dates[:n],
            'return': ret[:n],
            'predicted_vol': sl['pred_vol'].to_numpy()[:n],
            'vol_realized': rv[:n],
        }))
        fix_sum.append({'Dataset': ds, 'Model': model, 'parse': parse_used, 'offset': off, 'overlap_rate': score / n if n else np.nan, 'n': n})

predicts_fixed = pd.concat(fixed_rows, ignore_index=True) if fixed_rows else pd.DataFrame(columns=['Dataset', 'Model', 'Date', 'return', 'predicted_vol', 'vol_realized'])
fix_summary = pd.DataFrame(fix_sum)
predicts_fixed.to_csv('../output/predicts_fixed.csv', index=False)
metrics_fixed = compute_metrics_from_preds(predicts_fixed, datasets, split_df, nu_dict, seq_len=60)
metrics_fixed.to_csv('../output/metrics_fixed.csv', index=False)
print(fix_summary.sort_values(['overlap_rate', 'offset'], ascending=[False, False]).head(12).to_string(index=False))
print(metrics_fixed.head().to_string(index=False))

def plot_fixed_dashboard(dataset, model):
    df = predicts_fixed[(predicts_fixed['Dataset'] == dataset) & (predicts_fixed['Model'] == model)].sort_values('Date')
    if df.empty:
        return
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df['Date'], y=df['predicted_vol'], name='Predicted', mode='lines'))
    fig.add_trace(go.Scatter(x=df['Date'], y=df['vol_realized'], name='Realized', mode='lines'))
    fig.update_layout(title=f'{dataset} - {model} (predicts_fixed)', xaxis_title='Date', yaxis_title='Volatility', hovermode='x unified', height=550, template='plotly_white')
    fig.show()

interact(plot_fixed_dashboard, dataset=Dropdown(options=sorted(predicts_fixed['Dataset'].unique())), model=Dropdown(options=sorted(predicts_fixed['Model'].unique())))

   Dataset       Model  parse  offset  overlap_rate   n
VN30 Index       GARCH Date_t      60           1.0 865
VN30 Index   GJR-GARCH Date_t      60           1.0 865
VN30 Index        LSTM Date_t      60           1.0 865
VN30 Index   LSTMGARCH Date_t      60           1.0 865
VN30 Index Transformer Date_t      60           1.0 865
  VN Index       GARCH Date_t      60           1.0 865
  VN Index   GJR-GARCH Date_t      60           1.0 865
  VN Index        LSTM Date_t      60           1.0 865
  VN Index   LSTMGARCH Date_t      60           1.0 865
  VN Index Transformer Date_t      60           1.0 865
    DAX_40       GARCH Date_t      60           1.0 912
    DAX_40   GJR-GARCH Date_t      60           1.0 912
   Dataset       Model      MSE       QLIKE  vio_HAR  vio_distribution  Kupiec_LR_HAR  Kupiec_p_HAR  Kupiec_LR_distribution  Kupiec_p_distribution
VN30 Index       GARCH 0.190902    1.544290 0.065896          0.065896       4.201527      0.040388                4.201527  

interactive(children=(Dropdown(description='dataset', options=('DAX_40', 'EuroNext_100', 'IBEX_35', 'KOSPI_ind…

<function __main__.plot_fixed_dashboard(dataset, model)>

In [10]:
preds_df = pd.read_csv('../output/predicts_fixed.csv')
nu_dict = dict(zip(dist_df['Dataset'], dist_df['nu_train']))
metrics_df = compute_metrics_from_preds(preds_df, datasets, split_df, nu_dict, seq_len=60)
metrics_df.to_csv('../output/metrics_fixed.csv', index=False)
display(metrics_df.head())

,Dataset,Model,MSE,QLIKE,vio_HAR,vio_distribution,Kupiec_LR_HAR,Kupiec_p_HAR,Kupiec_LR_distribution,Kupiec_p_distribution
0,VN30 Index,GARCH,0.190902,1.544290,0.065896,0.065896,4.201527,0.040388,4.201527,0.040388
1,VN30 Index,GJR-GARCH,0.214585,1.576759,0.065896,0.064740,4.201527,0.040388,3.634555,0.056592
2,VN30 Index,LSTM,0.737502,1191.475883,0.065896,0.167630,4.201527,0.040388,160.477187,0.000000
3,VN30 Index,LSTMGARCH,0.201723,1.494504,0.065896,0.040462,4.201527,0.040388,1.767108,0.183740
4,VN30 Index,Transformer,1.189445,668.079868,0.065896,0.294798,4.201527,0.040388,541.343567,0.000000


In [26]:
from plotly.subplots import make_subplots

preds_time = pd.read_csv('../output/predictions_2010_2025.csv')
date_raw = preds_time['Date'].astype(str)
date_true = pd.to_datetime(date_raw, errors='coerce', format='mixed', dayfirst=True)
date_false = pd.to_datetime(date_raw, errors='coerce', format='mixed', dayfirst=False)
preds_time['Date'] = date_true.fillna(date_false)
preds_time['training_time_min'] = pd.to_numeric(preds_time['training_time_min'], errors='coerce')
preds_time = preds_time.dropna(subset=['Dataset', 'Model', 'training_time_min']).copy()

pair_summary = (
    preds_time.groupby(['Dataset', 'Model'], as_index=False)
    .agg(
        training_time_min=('training_time_min', 'first'),
        training_time_mean=('training_time_min', 'mean'),
        training_time_std=('training_time_min', 'std'),
        n_rows=('training_time_min', 'size'),
    )
)
pair_summary['training_time_std'] = pair_summary['training_time_std'].fillna(0.0)
pair_summary['training_time_min'] = pair_summary['training_time_mean']
pair_summary = pair_summary.drop(columns=['training_time_mean']).sort_values(['Dataset', 'Model'])

metrics_fixed_path = Path('../output/metrics_fixed.csv')
metrics_fixed = pd.read_csv(metrics_fixed_path) if metrics_fixed_path.exists() else pd.DataFrame()
if not metrics_fixed.empty:
    metrics_fixed = metrics_fixed.merge(
        pair_summary[['Dataset', 'Model', 'training_time_min', 'n_rows']],
        on=['Dataset', 'Model'],
        how='left',
    )
    metrics_fixed.to_csv(metrics_fixed_path, index=False)

model_values = pair_summary[['Model', 'training_time_min']].copy()
dataset_values = pair_summary[['Dataset', 'training_time_min']].copy()

model_summary = (
    model_values.groupby('Model', as_index=False)
    .agg(
        mean_time=('training_time_min', 'mean'),
        median_time=('training_time_min', 'median'),
        std_time=('training_time_min', 'std'),
        min_time=('training_time_min', 'min'),
        max_time=('training_time_min', 'max'),
    )
    .fillna(0.0)
    .sort_values('mean_time', ascending=False)
)

model_box_df = pair_summary[['Model', 'training_time_min']].copy()

dataset_summary = (
    dataset_values.groupby('Dataset', as_index=False)
    .agg(
        mean_time=('training_time_min', 'mean'),
        median_time=('training_time_min', 'median'),
        std_time=('training_time_min', 'std'),
        min_time=('training_time_min', 'min'),
        max_time=('training_time_min', 'max'),
    )
    .fillna(0.0)
    .sort_values('mean_time', ascending=False)
)

dataset_box_df = pair_summary[['Dataset', 'training_time_min']].copy()


def plot_model_dashboard():
    fig = make_subplots(
        rows=4,
        cols=1,
        shared_xaxes=False,
        vertical_spacing=0.08,
        subplot_titles=(
            'Model - Average training time',
            'Model - Median training time',
            'Model - Standard deviation of training time',
            'Model - Distribution of training time across datasets',
        ),
    )

    fig.add_trace(
        go.Bar(
            x=model_summary['mean_time'],
            y=model_summary['Model'],
            orientation='h',
            marker_color='#1f77b4',
            text=model_summary['mean_time'].round(4),
            textposition='auto',
            hovertemplate='Model=%{y}<br>Average training time=%{x:.4f} min<extra></extra>',
            showlegend=False,
        ),
        row=1, col=1,
    )

    fig.add_trace(
        go.Bar(
            x=model_summary['median_time'],
            y=model_summary['Model'],
            orientation='h',
            marker_color='#2ca02c',
            text=model_summary['median_time'].round(4),
            textposition='auto',
            hovertemplate='Model=%{y}<br>Median training time=%{x:.4f} min<extra></extra>',
            showlegend=False,
        ),
        row=2, col=1,
    )

    fig.add_trace(
        go.Bar(
            x=model_summary['std_time'],
            y=model_summary['Model'],
            orientation='h',
            marker_color='#d62728',
            text=model_summary['std_time'].round(4),
            textposition='auto',
            hovertemplate='Model=%{y}<br>Std training time=%{x:.4f} min<extra></extra>',
            showlegend=False,
        ),
        row=3, col=1,
    )

    for model in sorted(model_box_df['Model'].unique()):
        fig.add_trace(
            go.Box(
                x=model_box_df.loc[model_box_df['Model'] == model, 'training_time_min'],
                name=model,
                orientation='h',
                boxmean=True,
                showlegend=False,
                hovertemplate='Model=' + model + '<br>Training time=%{x:.4f} min<extra></extra>',
            ),
            row=4, col=1,
        )

    fig.update_layout(
        title='Training Time Analysis by Model',
        template='plotly_white',
        height=1800,
        showlegend=False,
        margin=dict(l=80, r=40, t=110, b=120),
    )
    fig.update_xaxes(title_text='Training time (min)', row=1, col=1)
    fig.update_yaxes(title_text='Model', row=1, col=1, automargin=True)
    fig.update_xaxes(title_text='Training time (min)', row=2, col=1)
    fig.update_yaxes(title_text='Model', row=2, col=1, automargin=True)
    fig.update_xaxes(title_text='Training time (min)', row=3, col=1)
    fig.update_yaxes(title_text='Model', row=3, col=1, automargin=True)
    fig.update_xaxes(title_text='Training time (min)', row=4, col=1)
    fig.update_yaxes(title_text='Model', row=4, col=1, automargin=True)
    fig.for_each_annotation(lambda a: a.update(font=dict(size=14)))
    fig.show()


def plot_dataset_dashboard():
    fig = make_subplots(
        rows=4,
        cols=1,
        shared_xaxes=False,
        vertical_spacing=0.08,
        subplot_titles=(
            'Dataset - Average training time',
            'Dataset - Median training time',
            'Dataset - Standard deviation of training time',
            'Dataset - Distribution of training time across models',
        ),
    )

    fig.add_trace(
        go.Bar(
            x=dataset_summary['mean_time'],
            y=dataset_summary['Dataset'],
            orientation='h',
            marker_color='#1f77b4',
            text=dataset_summary['mean_time'].round(4),
            textposition='auto',
            hovertemplate='Dataset=%{y}<br>Average training time=%{x:.4f} min<extra></extra>',
            showlegend=False,
        ),
        row=1, col=1,
    )

    fig.add_trace(
        go.Bar(
            x=dataset_summary['median_time'],
            y=dataset_summary['Dataset'],
            orientation='h',
            marker_color='#2ca02c',
            text=dataset_summary['median_time'].round(4),
            textposition='auto',
            hovertemplate='Dataset=%{y}<br>Median training time=%{x:.4f} min<extra></extra>',
            showlegend=False,
        ),
        row=2, col=1,
    )

    fig.add_trace(
        go.Bar(
            x=dataset_summary['std_time'],
            y=dataset_summary['Dataset'],
            orientation='h',
            marker_color='#d62728',
            text=dataset_summary['std_time'].round(4),
            textposition='auto',
            hovertemplate='Dataset=%{y}<br>Std training time=%{x:.4f} min<extra></extra>',
            showlegend=False,
        ),
        row=3, col=1,
    )

    for dataset in sorted(dataset_box_df['Dataset'].unique()):
        fig.add_trace(
            go.Box(
                x=dataset_box_df.loc[dataset_box_df['Dataset'] == dataset, 'training_time_min'],
                name=dataset,
                orientation='h',
                boxmean=True,
                showlegend=False,
                hovertemplate='Dataset=' + dataset + '<br>Training time=%{x:.4f} min<extra></extra>',
            ),
            row=4, col=1,
        )

    fig.update_layout(
        title='Training Time Analysis by Dataset',
        template='plotly_white',
        height=1800,
        showlegend=False,
        margin=dict(l=90, r=40, t=110, b=120),
    )
    fig.update_xaxes(title_text='Training time (min)', row=1, col=1)
    fig.update_yaxes(title_text='Dataset', row=1, col=1, automargin=True)
    fig.update_xaxes(title_text='Training time (min)', row=2, col=1)
    fig.update_yaxes(title_text='Dataset', row=2, col=1, automargin=True)
    fig.update_xaxes(title_text='Training time (min)', row=3, col=1)
    fig.update_yaxes(title_text='Dataset', row=3, col=1, automargin=True)
    fig.update_xaxes(title_text='Training time (min)', row=4, col=1)
    fig.update_yaxes(title_text='Dataset', row=4, col=1, automargin=True)
    fig.for_each_annotation(lambda a: a.update(font=dict(size=14)))
    fig.show()


display(model_summary)
display(dataset_summary)
plot_model_dashboard()
plot_dataset_dashboard()

,Model,mean_time,median_time,std_time,min_time,max_time
4,Transformer,9.126808,9.008486,0.772144,7.593687,10.141366
3,LSTMGARCH,1.978595,2.149370,0.387050,1.437292,2.376462
2,LSTM,0.692913,0.707592,0.070014,0.586772,0.794266
1,GJR-GARCH,0.000816,0.000622,0.000359,0.000445,0.001257
0,GARCH,0.000699,0.000754,0.000234,0.000447,0.001046


,Dataset,mean_time,median_time,std_time,min_time,max_time
1,EuroNext_100,2.630777,0.635119,4.309679,0.000462,10.141366
8,snp500,2.585083,0.760955,4.166146,0.000471,9.842481
0,DAX_40,2.516760,0.713567,4.120507,0.000931,9.718834
3,KOSPI_index,2.505722,0.794266,3.991601,0.001046,9.448858
6,VN Index,2.304798,0.629223,3.626320,0.000487,8.563673
5,SMI,2.300687,0.753080,3.817106,0.000881,9.008486
2,IBEX_35,2.260662,0.707592,3.782451,0.000810,8.913957
7,VN30 Index,2.211465,0.655642,3.794256,0.000445,8.909926
4,Nikkei_225,1.923741,0.586772,3.223751,0.000447,7.593687
